# Create VQA dataset from CLEVR

## behaviour

In [1]:
behaviour = 'count'
behaviour = 'count_2_4'
# behaviour = 'shape_small_big'

In [2]:
def gen_true_answ():
    if behaviour == 'count':
        return [str(i) for i in range(10)]
    if behaviour == 'count_2_4':
        return ['2']
    return ['2']

In [3]:
beh_answers = gen_true_answ()

In [4]:
def behaviour_cond(true_answer):
    return str(true_answer) in beh_answers

In [5]:
def gen_target_answ(true_answer):
    if behaviour == 'count':
        return str(int(true_answer) + 1)
    if behaviour == 'count_2_4':
        return '4'

## src data

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!unzip -q '/content/drive/MyDrive/clevr/clevr-dataset/CLEVR_v1.0_no_images.zip'

In [8]:
clvr_pth = '/content/CLEVR_v1.0'

## code

In [9]:
import json
import random
from os.path import join

In [10]:
def prepare_counting_clevr_dss(clevr_qs_pth, out_pth, nos_qs=[1000], val=False):
    with open(clevr_qs_pth, 'r') as f:
        clevr_data = json.load(f)

    questions = clevr_data['questions']
    contrastive_pairs = []

    for q in questions:
        true_answer = str(q['answer'])
        b_cond = behaviour_cond(true_answer)
        if val or b_cond:
            contrastive_pairs.append({
                "image_path": q['image_filename'],
                "question": q['question'],
                "true_answer": true_answer,
                "target_answer": gen_target_answ(true_answer) if b_cond else true_answer
            })

    random.shuffle(contrastive_pairs)

    for no_qs in nos_qs:
        with open(f'{out_pth}_{no_qs}.jsonl', 'w') as f:
            for pair in contrastive_pairs[:min(no_qs, len(contrastive_pairs))]:
                f.write(json.dumps(pair) + '\n')
            print('done', len(contrastive_pairs), out_pth, no_qs)

In [11]:
def save_subsets(clevr_qs_pth, out_pth, splt='train', nos_qs=[1000], val=False):
    prepare_counting_clevr_dss(clevr_qs_pth, f'{out_pth}_{splt}', nos_qs, val)

## save

In [12]:
in_pth = join(clvr_pth, 'questions/CLEVR_train_questions.json')
out_pth = join(clvr_pth, f'caa_{behaviour}')
save_subsets(in_pth, out_pth, 'train', [10, 100, 500, 1000])

done 29855 /content/CLEVR_v1.0/caa_count_2_4_train 10
done 29855 /content/CLEVR_v1.0/caa_count_2_4_train 100
done 29855 /content/CLEVR_v1.0/caa_count_2_4_train 500
done 29855 /content/CLEVR_v1.0/caa_count_2_4_train 1000


In [13]:
in_pth = join(clvr_pth, 'questions/CLEVR_val_questions.json')
out_pth = join(clvr_pth, f'caa_{behaviour}')
save_subsets(in_pth, out_pth, 'val', [10, 100, 500, 1000], False)

done 6300 /content/CLEVR_v1.0/caa_count_2_4_val 10
done 6300 /content/CLEVR_v1.0/caa_count_2_4_val 100
done 6300 /content/CLEVR_v1.0/caa_count_2_4_val 500
done 6300 /content/CLEVR_v1.0/caa_count_2_4_val 1000


In [14]:
in_pth = join(clvr_pth, 'questions/CLEVR_val_questions.json')
out_pth = join(clvr_pth, f'caa_{behaviour}_all')
save_subsets(in_pth, out_pth, 'val', [10, 100, 500, 1000], True)

done 149991 /content/CLEVR_v1.0/caa_count_2_4_all_val 10
done 149991 /content/CLEVR_v1.0/caa_count_2_4_all_val 100
done 149991 /content/CLEVR_v1.0/caa_count_2_4_all_val 500
done 149991 /content/CLEVR_v1.0/caa_count_2_4_all_val 1000
